# 02 · Amostras estáveis

**O que esta etapa faz:** sorteia pontos de treino onde as coleções do MapBiomas concordam sobre a classe, ano a ano e ao longo de uma janela de anos. As classes são reagrupadas em 4: **1 urbano, 2 vegetação, 3 água, 4 agropecuária**. São sorteados N pontos por classe em cada *super carta* (a carta e suas vizinhas). Depois os pontos são separados em treino (80%) e validação (20%).

**Equivalente na Col 11 (versão de produção):** `collection 11/codes_producao/01_amostras/`: `samplesCreation_v4.js`, `mb_batch/collections.js` e `visSamplesByGrid_v2.js`.

**Parâmetros** em `config/col12.yaml`, bloco `samples`: coleções usadas na estabilidade, janelas de anos, pontos por classe, semente e fração de treino.

Nada é exportado sem você rodar explicitamente o bloco 5.

## 0 · Preparação

In [ ]:
import sys, os
EM_COLAB = 'google.colab' in sys.modules
if EM_COLAB:
    !git clone -q -b col12 https://github.com/jpedrassoli/brazil-urban-area.git repo 2>/dev/null || (cd repo && git pull -q)
    %pip install -q -e "repo/lulc_30m_landsat/collection 12" geemap
    RAIZ = 'repo/lulc_30m_landsat/collection 12'
else:
    RAIZ = os.path.abspath('..')
sys.path.insert(0, RAIZ)

import ee
from mb_urban import auth, config, grids, mosaic, samples, classify, tasks, col11
cfg = config.load()

# --- Sua configuração -------------------------------------------------------------------------
# Usada quando não há um config/col12.yaml próprio: preencha com o SEU projeto do Cloud registrado no
# Earth Engine. A pasta de destino precisa existir; a tabela de amostras é o CSV de
# `collection 11/dados` subido no seu Earth Engine.
if config.config_path() == config.EXAMPLE_CONFIG:
    PROJETO = 'seu-projeto-do-cloud'
    cfg['cloud_project'] = PROJETO
    cfg['destinations']['teste'] = f'projects/{PROJETO}/assets/mb_urban_col12'
    cfg['samples']['table'] = f'projects/{PROJETO}/assets/amostras_col11'

## 1 · Conta do Earth Engine

**No Colab**, aparece um **link**: abra, entre com a sua conta do Earth Engine, autorize, copie o código que o Google mostra e cole no campo abaixo do bloco. O login usa o cliente do próprio Google Cloud (gcloud) e funciona também com contas institucionais. A credencial fica só na máquina temporária do Colab e some quando o ambiente é encerrado. **Não compartilhe o código de verificação.**

O aviso "Cannot find a quota project" pode ser ignorado: o projeto do Cloud é informado em seguida pelo notebook.

**No computador**, usa o login padrão do Earth Engine (só pede login se ainda não houver credencial).

In [ ]:
CONTA = 'default'
if CONTA == 'default':
    if EM_COLAB:
        # Login pelo gcloud (cliente de login do próprio Google). Aparece um link: abra, entre com a
        # sua conta, autorize, copie o código e cole no campo abaixo deste bloco. A credencial fica só
        # na máquina temporária do Colab. Não compartilhe o código com ninguém.
        adc = os.path.join(os.environ.get('CLOUDSDK_CONFIG', os.path.expanduser('~/.config/gcloud')),
                           'application_default_credentials.json')
        if not os.path.exists(adc):
            !gcloud auth application-default login --no-launch-browser --scopes=https://www.googleapis.com/auth/earthengine,https://www.googleapis.com/auth/cloud-platform
    else:
        ee.Authenticate()   # só pede login se ainda não houver credencial válida
auth.init(CONTA)

## 2 · Parâmetros
Carta e janela de anos. As janelas disponíveis saem da configuração.

In [ ]:
CARTA = 'SF-23-Y-C'
janelas = samples.windows(cfg)
print({k: f'{v[0]}-{v[-1]} ({len(v)} anos)' for k, v in janelas.items()})
JANELA = list(janelas)[-1]
print('janela escolhida:', JANELA)

## 3 · Visualização das áreas estáveis
Classes estáveis na janela, sobre a super carta. Cores: urbano (vermelho), vegetação (verde), água (azul), agropecuária (amarelo).

In [ ]:
import geemap
estavel = samples.stable_classes(JANELA, cfg)
ctx = grids.context(CARTA, cfg)
m = geemap.Map(); m.centerObject(ctx, 9)
m.addLayer(estavel.clip(ctx), {'min': 1, 'max': 4, 'palette': ['#c0392b', '#27ae60', '#2e86de', '#f1c40f']}, 'classes estáveis')
m.addLayer(grids.context_fc(CARTA, cfg).style(color='white', fillColor='00000000'), {}, 'super carta')
m

## 4 · Amostras usadas pela classificação
A classificação lê a tabela `samples.table` da configuração. Para reproduzir a Col 11, ela aponta para a tabela de produção da Col 11. Contagem na super carta, na janela escolhida:

In [ ]:
tab = samples.table(cfg).filterBounds(ctx).filter(ee.Filter.eq('range', JANELA))
print('por classe:', tab.aggregate_histogram('class').getInfo())
print('por tipo:', tab.aggregate_histogram('type').getInfo())

## 5 · (Opcional) Gerar novas amostras
Sorteia as amostras da carta em todas as janelas e grava como tabela no destino (pasta `amostras/`), já separadas em treino e validação.
**Só cria e inicia a tarefa quando `EXPORTAR = True`.**

In [ ]:
EXPORTAR = False
if EXPORTAR:
    fc = ee.FeatureCollection([samples.create(CARTA, j, cfg) for j in janelas]).flatten()
    fc = samples.split_train_validation(fc, cfg)
    nome = f'amostras_{CARTA}'
    asset = config.asset_path('amostras', nome, cfg=cfg)
    spec = [(nome, asset, lambda: ee.batch.Export.table.toAsset(collection=fc, description=config.task_description(nome, cfg), assetId=asset))]
    print(tasks.run_queue(spec, account=CONTA, log_name='amostras', cfg=cfg))
else:
    print('exportação desligada (EXPORTAR = False)')

## 6 · Acompanhar tarefas

In [ ]:
tasks.my_tasks(CONTA)